# Hackathon · Team 4 · Model v2: a better layout generator (train on Colab)

Model v1 (`04_layout_generator.ipynb`) works, but its pictures are often not usable: in the app, step 3 drops many of them, mostly because **the number of bedrooms is wrong** (right in only 37% of the raw pictures). v2 keeps the same data, the same inputs and the same outputs (so the server and steps 3-4 do not change), and changes the model and the training:

| | v1 | v2 |
|---|---|---|
| the request (bedrooms, balcony, area) | enters the decoder once, at the start | reaches **every layer** of the decoder (FiLM: it scales and shifts the features) |
| decoder width at 8-32 px | 128 / 64 channels, one convolution per step | 192 / 96 channels, two convolutions per step |
| bedroom count | only through the room maps | plus a **bedroom checker**: a small network trained on real plans counts the bedrooms in what the decoder draws, and the decoder is pushed to draw what was asked |
| kitchen in the loss | weight 2 | weight 3 (the kitchen was often swallowed by the living room) |
| training | CPU, up to 40 epochs (68 minutes) | **GPU** (Colab), up to 120 epochs, the learning rate goes down slowly |

Both models are compared at the end on the **same requests**: raw pictures (bedrooms right, balcony right, different options) and, most important, **how many pictures step 3 can use**. The server switches to v2 only if v2 is better.

### Before you run it (5 minutes)
1. In Google Drive, make a folder **`team4_v2`** and put these 6 files in it (from the repository on your computer):
   - `data/generator_maps_128px.npz` and `data/generator_lib.csv`
   - `01_autoencoder/models/layout_generator/layout_cvae.pt`, `layout_cvae_config.json`, `room_rules.json` (v1, for the comparison)
   - `01_autoencoder/notebook/layout_core.py`
2. Open this notebook in Colab: **File → Upload notebook**.
3. **Runtime → Change runtime type → T4 GPU**.
4. **Runtime → Run all**. Colab asks for access to your Drive: allow it.

Everything the notebook makes is saved in `team4_v2/v2/` on your Drive, **after every epoch**: if Colab disconnects, run all again and the training continues where it stopped. Keep the Colab tab open while it trains.

# 0. Setup

In [ ]:
import json
import os
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/team4_v2")              # the 6 files are here
    DATA_DIR = MODEL_V1_DIR = CODE_DIR = WORK
    OUT_DIR = WORK / "v2"                                        # everything this notebook makes
    os.system("pip install -q onnx onnxruntime")
else:                                                            # on your computer, from 01_autoencoder/notebook
    DATA_DIR = Path("../../data")
    MODEL_V1_DIR = Path("../models") / "layout_generator"
    CODE_DIR = Path(".")
    OUT_DIR = Path("../models") / "layout_generator_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR if IN_COLAB else Path("../figures")
needed = [DATA_DIR / "generator_maps_128px.npz", DATA_DIR / "generator_lib.csv", MODEL_V1_DIR / "layout_cvae.pt",
          MODEL_V1_DIR / "layout_cvae_config.json", MODEL_V1_DIR / "room_rules.json", CODE_DIR / "layout_core.py"]
missing = [str(p) for p in needed if not p.exists()]
assert not missing, f"missing files: {missing}"

sys.path.insert(0, str(CODE_DIR))
import layout_core as lc
assert hasattr(lc, "LayoutCVAE2"), "layout_core.py is too old: copy the latest one"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
print(f"PyTorch {torch.__version__} | device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else
      "  <- no GPU: in Colab, Runtime > Change runtime type > T4 GPU"))


def save_fig(name):
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Figure saved to {path}")

# 1. The data (the same as v1)

The room maps made by `04_layout_generator.ipynb` (15,832 plans, 8 layers of 128 × 128) and the plan table made by `05_layout_to_vector.ipynb` (split, bedrooms, balcony, scale). The condition uses v1's area numbers, so v1 and v2 read the same request the same way.

In [ ]:
cache = np.load(DATA_DIR / "generator_maps_128px.npz")
sil_u8, maps_u8 = cache["sil"], cache["maps"]                       # read once (each read of the npz unpacks it again)
scales, sil_units2 = cache["scales"], cache["sil_units2"]
lib = pd.read_csv(DATA_DIR / "generator_lib.csv")
assert np.array_equal(lib["index"].to_numpy(), cache["index"]), "generator_lib.csv does not match the room maps"
lib["area_m2"] = sil_units2 * lib["m_per_unit"] ** 2
lib["m_per_px"] = lib["m_per_unit"] * (512 / lc.IMG) / scales

config_v1 = json.loads((MODEL_V1_DIR / "layout_cvae_config.json").read_text())
AREA_MEAN, AREA_STD = config_v1["condition"]["area_mean"], config_v1["condition"]["area_std"]
rules = json.loads((MODEL_V1_DIR / "room_rules.json").read_text())
conds = np.stack([lc.condition_vector(b, y, a, AREA_MEAN, AREA_STD)
                  for b, y, a in zip(lib["bedrooms"], lib["balcony"], lib["area_m2"])])
bed_class = np.clip(lib["bedrooms"].to_numpy(), 1, lc.BED_MAX) - 1          # 0..4 = 1..5+ bedrooms

is_train, is_val = (lib["split"] == "train").to_numpy(), (lib["split"] == "val").to_numpy()
train_rows, val_rows = np.where(is_train)[0], np.where(is_val)[0]
print(f"{len(lib):,} plans | train {len(train_rows):,} | validation {len(val_rows):,} | room maps {maps_u8.nbytes / 1e9:.1f} GB")

# On the GPU the training plans stay as 0-255 bytes (1.7 GB); each batch becomes 0-1 there: no copying every batch
store = "cuda" if DEVICE == "cuda" else "cpu"
X_train = torch.from_numpy(maps_u8[train_rows]).to(store)
S_train = torch.from_numpy(sil_u8[train_rows]).unsqueeze(1).to(store)
C_train = torch.from_numpy(conds[train_rows]).to(store)
B_train = torch.from_numpy(bed_class[train_rows]).long().to(store)
val_set = tuple(torch.from_numpy(a) for a in (maps_u8[val_rows], sil_u8[val_rows][:, None], conds[val_rows]))
score_rows = np.random.default_rng(0).choice(val_rows, size=min(100, len(val_rows)), replace=False)


def batch(idx):
    # one training batch, 0-1, with a random mirror (the same for maps and silhouette): left-right, up-down or both
    x = X_train[idx].to(DEVICE).float() / 255
    s = S_train[idx].to(DEVICE).float() / 255
    if torch.rand(1).item() < 0.5:
        x, s = x.flip(3), s.flip(3)
    if torch.rand(1).item() < 0.5:
        x, s = x.flip(2), s.flip(2)
    return x, s, C_train[idx].to(DEVICE), B_train[idx].to(DEVICE)

# 2. The model, the bedroom checker and the loss

**v2** is `LayoutCVAE2` in `layout_core.py` (the server and the notebooks use the same file). **The bedroom checker** is only for training: a small network that looks at a room map and says how many bedrooms it has (1, 2, 3, 4, 5+). It learns on the real plans. Then, for every batch, the decoder draws a plan from **random numbers** (exactly as in the app) and the checker counts its bedrooms: if the count is not the one asked, the decoder is pushed to fix it (`AUX_WEIGHT`).

In [ ]:
#                          living bedroom bathroom kitchen balcony wall door window
CH_WEIGHTS = torch.tensor([1.0,   1.0,    2.0,     3.0,    2.0,    3.0, 8.0, 8.0], device=DEVICE).view(1, -1, 1, 1)
FREE_BITS = 0.1


def cvae_loss(out, x, mu, logvar, beta):
    # the same loss as v1 (BCE + Dice + beta * KL with free bits), the kitchen weighs a bit more
    bce = (F.binary_cross_entropy(out, x, reduction="none") * CH_WEIGHTS).mean()
    inter = (out * x).sum(dim=(2, 3))
    total = out.sum(dim=(2, 3)) + x.sum(dim=(2, 3))
    dice = (1 - (2 * inter + 1) / (total + 1)).mean()
    kl_per_number = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp())).mean(0)
    kl_loss = kl_per_number.clamp(min=FREE_BITS).sum()
    return bce + dice + beta * kl_loss, bce.item(), dice.item(), kl_per_number.sum().item()


class BedroomChecker(nn.Module):
    # room map (8 x 128 x 128) -> how many bedrooms: 5 numbers (1, 2, 3, 4, 5+)
    def __init__(self, n_in=len(lc.CHANNELS)):
        super().__init__()
        block = lambda a, b: nn.Sequential(nn.Conv2d(a, b, 3, stride=2, padding=1), nn.BatchNorm2d(b), nn.ReLU())
        self.net = nn.Sequential(block(n_in, 32), block(32, 64), block(64, 96), block(96, 128), block(128, 128),
                                 nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, lc.BED_MAX))

    def forward(self, maps):
        return self.net(maps)


def n_params(m):
    return sum(p.numel() for p in m.parameters())


v2_test = lc.LayoutCVAE2()
out, mu, logvar = v2_test(torch.zeros(2, 8, 128, 128), torch.ones(2, 1, 128, 128), torch.zeros(2, 7))
print(f"v1: {n_params(lc.LayoutCVAE()):,} parameters | v2: {n_params(v2_test):,} parameters | output {tuple(out.shape)}")
print(f"bedroom checker: {n_params(BedroomChecker()):,} parameters (training only)")

# 3. Scoring (the same as v1)

Every epoch: 100 validation silhouettes, each with its real request, 3 invented options. **Bedrooms right** and **balcony right** are counted in the raw pictures, exactly as in `04`. Score = bedrooms right + balcony right (−1 if the options are almost the same). The best epoch is kept.

In [ ]:
def count_in_map(maps, m_per_px, min_bedroom_m2=6.0, min_balcony_m2=1.0):
    rooms = maps[:len(lc.ROOMS)]
    label, strong = rooms.argmax(0), rooms.max(0) > 0.5
    not_wall = maps[lc.CHANNELS.index("wall")] < 0.35
    bed = ((label == lc.ROOMS.index("bedroom")) & strong & not_wall).astype(np.uint8)
    _, _, stats, _ = cv2.connectedComponentsWithStats(bed, connectivity=4)
    bedrooms = int((stats[1:, cv2.CC_STAT_AREA] * m_per_px ** 2 >= min_bedroom_m2).sum())
    balcony_m2 = ((label == lc.ROOMS.index("balcony")) & strong).sum() * m_per_px ** 2
    return bedrooms, int(balcony_m2 >= min_balcony_m2)


def label_map(maps, sil):
    label = np.where(maps[:len(lc.ROOMS)].max(0) > 0.5, maps[:len(lc.ROOMS)].argmax(0), -1)
    return label[np.asarray(sil) > 127]


def request_score(model, rows, n=3):
    beds_ok, balcony_ok, different = [], [], []
    for k, r in enumerate(rows):
        options = lc.generate(model, sil_u8[r], conds[r], n=n, seed=k, device=DEVICE)
        for o in options:
            beds, balcony = count_in_map(o, lib.loc[r, "m_per_px"])
            beds_ok.append(min(beds, lc.BED_MAX) == bed_class[r] + 1)
            balcony_ok.append(balcony == lib.loc[r, "balcony"])
        labels = [label_map(o, sil_u8[r]) for o in options]
        different.append(np.mean([(labels[i] != labels[j]).mean() for i in range(n) for j in range(i + 1, n)]))
    result = {"bedrooms_right": float(np.mean(beds_ok)), "balcony_right": float(np.mean(balcony_ok)),
              "different": float(np.mean(different))}
    result["score"] = result["bedrooms_right"] + result["balcony_right"] - (1.0 if result["different"] < 0.05 else 0.0)
    return result


@torch.no_grad()
def evaluate(model, data, batch_size=500):
    model.eval()
    sums, n = np.zeros(3), 0
    for i in range(0, len(data[0]), batch_size):
        x, s, c = (t[i:i + batch_size].to(DEVICE).float() for t in data)
        x, s = x / 255, s / 255
        mu, logvar = model.encode(x, s, c)
        _, bce, dice, kl = cvae_loss(model.decode(mu, s, c), x, mu, logvar, 0.0)
        sums += np.array([bce, dice, kl]) * len(x)
        n += len(x)
    return dict(zip(["val_bce", "val_dice", "val_kl"], sums / n))

# 4. Training

1. **The bedroom checker** learns first, on the real plans (3 epochs, about a minute on a GPU).
2. **A short trial** chooses `AUX_WEIGHT` (how hard the checker pushes): 0 (no push, like v1), 0.3 or 1.0, 4 epochs each on 40% of the plans. The best score wins.
3. **The full training**: up to `MAX_EPOCHS`, it stops after `PATIENCE` epochs without a better score. `BETA` = 0.008, the value v1 chose.

Everything is saved after each epoch (`OUT_DIR/checkpoint.pt`): run the cell again and it continues.

In [ ]:
BETA = 0.008
AUX_WEIGHTS = [0.0, 0.3, 1.0]
TRIAL_EPOCHS, TRIAL_SHARE = 4, 0.4
MAX_EPOCHS, PATIENCE = 120, 15
BATCH, LR = 128, 1e-3
WEIGHTS_V2 = OUT_DIR / "layout_cvae_v2.pt"
CHECKPOINT = OUT_DIR / "checkpoint.pt"
TRAIN_LOG = OUT_DIR / "layout_cvae_v2_training.json"
CHECKER = OUT_DIR / "bedroom_checker.pt"


def train_checker(epochs=3):
    checker = BedroomChecker().to(DEVICE)
    opt = torch.optim.Adam(checker.parameters(), lr=1e-3)
    for epoch in range(epochs):
        checker.train()
        for idx in torch.randperm(len(X_train), device=store).split(BATCH):
            x, s, c, b = batch(idx)
            loss = F.cross_entropy(checker(x), b)
            opt.zero_grad()
            loss.backward()
            opt.step()
    checker.eval()
    with torch.no_grad():                                     # accuracy on validation plans
        right = []
        for i in range(0, len(val_rows), 500):
            x = val_set[0][i:i + 500].to(DEVICE).float() / 255
            right.append((checker(x).argmax(1).cpu().numpy() == bed_class[val_rows[i:i + 500]]))
    print(f"bedroom checker: right on {np.concatenate(right).mean():.0%} of the real validation plans")
    return checker


def train_v2(rows, epochs, aux_weight, checker, patience=None, save=False, warmup=5, verbose=True):
    torch.manual_seed(RANDOM_STATE)
    model = lc.LayoutCVAE2().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=LR / 20)
    for p in checker.parameters():
        p.requires_grad_(False)                                # the checker only judges; it does not learn here
    history, best, best_epoch, first = [], -np.inf, 0, 1
    if save and CHECKPOINT.exists():                           # continue after a disconnection
        ck = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)     # our own file
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        history, best, best_epoch, first = ck["history"], ck["best"], ck["best_epoch"], ck["epoch"] + 1
        print(f"continuing after epoch {ck['epoch']} (best so far: epoch {best_epoch})")
    rows_t = torch.as_tensor(rows, device=store)
    start = time.time()
    for epoch in range(first, epochs + 1):
        if patience and epoch - max(best_epoch, warmup) > patience:
            break
        b = BETA * min(1.0, epoch / warmup)
        model.train()
        total = 0.0
        for idx in rows_t[torch.randperm(len(rows_t), device=store)].split(BATCH):
            x, s, c, beds = batch(idx)
            out, mu, logvar = model(x, s, c)
            loss, *_ = cvae_loss(out, x, mu, logvar, b)
            if aux_weight > 0:                                  # draw from random numbers, as the app does
                drawn = model.decode(torch.randn_like(mu), s, c)
                loss = loss + aux_weight * F.cross_entropy(checker(drawn), beds)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        sched.step()
        row = {"epoch": epoch, "beta": b, "lr": sched.get_last_lr()[0], "train_loss": total / len(rows),
               **evaluate(model, val_set), **request_score(model, score_rows)}
        row = {k: (int(v) if k == "epoch" else float(v)) for k, v in row.items()}
        history.append(row)
        if verbose:
            print(f"epoch {epoch:3d}  val BCE {row['val_bce']:.4f}  Dice {row['val_dice']:.3f}  KL {row['val_kl']:5.1f}  |  "
                  f"bedrooms right {row['bedrooms_right']:.0%}  balcony right {row['balcony_right']:.0%}  "
                  f"different {row['different']:.0%}  score {row['score']:.2f}  ({time.time() - start:.0f} s)")
            if epoch == first:
                print(f"   -> about {(time.time() - start) * (epochs - first + 1) / 60:.0f} minutes for all epochs (less if it stops early)")
        if epoch >= warmup and row["score"] > best:
            best, best_epoch = row["score"], epoch
            if save:
                torch.save(model.state_dict(), WEIGHTS_V2)
        if save:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "history": history, "best": best, "best_epoch": best_epoch, "epoch": epoch}, CHECKPOINT)
    if save and WEIGHTS_V2.exists():
        model.load_state_dict(torch.load(WEIGHTS_V2, map_location=DEVICE))   # keep the best epoch
    return model, history, best_epoch

In [ ]:
RETRAIN = False          # False: if v2 is already trained (TRAIN_LOG exists), load it instead of training again

if TRAIN_LOG.exists() and WEIGHTS_V2.exists() and not RETRAIN:
    log = json.loads(TRAIN_LOG.read_text())
    AUX_WEIGHT, history, best_epoch, trial_table = log["aux_weight"], log["history"], log["best_epoch"], pd.DataFrame(log["trials"])
    v2 = lc.LayoutCVAE2().to(DEVICE)
    v2.load_state_dict(torch.load(WEIGHTS_V2, map_location=DEVICE))
    print(f"loaded v2: AUX_WEIGHT {AUX_WEIGHT}, best epoch {best_epoch} of {len(history)}")
else:
    start = time.time()
    if CHECKER.exists():
        checker = BedroomChecker().to(DEVICE)
        checker.load_state_dict(torch.load(CHECKER, map_location=DEVICE))
        checker.eval()
    else:
        checker = train_checker()
        torch.save(checker.state_dict(), CHECKER)
    trials_file = OUT_DIR / "trials.json"
    trials = json.loads(trials_file.read_text()) if trials_file.exists() else []
    trial_rows = np.random.default_rng(1).choice(len(train_rows), size=int(TRIAL_SHARE * len(train_rows)), replace=False)
    for w in AUX_WEIGHTS:
        if any(t["AUX_WEIGHT"] == w for t in trials):
            continue
        _, h, _ = train_v2(trial_rows, TRIAL_EPOCHS, w, checker, warmup=2, verbose=False)
        trials.append({"AUX_WEIGHT": w, **{k: h[-1][k] for k in ["val_bce", "val_dice", "bedrooms_right",
                                                                 "balcony_right", "different", "score"]}})
        trials_file.write_text(json.dumps(trials, indent=1, default=float))
        print(f"trial AUX_WEIGHT {w}: score {trials[-1]['score']:.2f}, bedrooms right {trials[-1]['bedrooms_right']:.0%}, "
              f"different {trials[-1]['different']:.0%}  ({time.time() - start:.0f} s)")
    trial_table = pd.DataFrame(trials)
    AUX_WEIGHT = float(trial_table.loc[trial_table["score"].idxmax(), "AUX_WEIGHT"])
    print(f"\nchosen AUX_WEIGHT = {AUX_WEIGHT}. Full training:")
    v2, history, best_epoch = train_v2(np.arange(len(train_rows)), MAX_EPOCHS, AUX_WEIGHT, checker, patience=PATIENCE, save=True)
    TRAIN_LOG.write_text(json.dumps({"beta": BETA, "aux_weight": AUX_WEIGHT, "best_epoch": best_epoch, "history": history,
                                     "trials": trials, "minutes": (time.time() - start) / 60}, indent=1, default=float))
    print(f"best epoch {best_epoch}, saved to {WEIGHTS_V2} ({(time.time() - start) / 60:.0f} minutes in total)")
trial_table.round(3)

In [ ]:
h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 4, figsize=(20, 3.8))
axes[0].plot(h["epoch"], h["val_bce"], label="validation BCE"); axes[0].plot(h["epoch"], h["val_dice"], label="validation Dice")
axes[0].legend(); axes[0].set_title("rebuilding error (lower = better)")
axes[1].plot(h["epoch"], h["val_kl"]); axes[1].set_title("KL (information in the code)")
axes[2].plot(h["epoch"], h["bedrooms_right"], label="bedrooms right"); axes[2].plot(h["epoch"], h["balcony_right"], label="balcony right")
axes[2].axhline(config_v1["check_on_validation"]["invented: bedrooms right"], color="#888780", ls="--", label="v1 bedrooms right")
axes[2].legend(); axes[2].set_ylim(0, 1); axes[2].set_title("does it draw what was asked?")
axes[3].plot(h["epoch"], h["score"]); axes[3].axvline(best_epoch, color="#e34948", ls=":"); axes[3].set_title(f"score (best epoch {best_epoch})")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout()
save_fig("70_v2_training")
plt.show()

# 5. v1 against v2, on the same requests

**A. The raw pictures** (300 validation requests, 3 options each): bedrooms right, balcony right, how different the options are.

**B. What matters for the app:** 40 validation requests, 30 pictures each, through **step 3** (`layout_core.make_layout`): how many pictures can be used, how many requests get 5 options, and why the others are dropped.

In [ ]:
v1 = lc.LayoutCVAE(latent_dim=config_v1["latent_dim"]).to(DEVICE)
v1.load_state_dict(torch.load(MODEL_V1_DIR / config_v1["weights"], map_location=DEVICE))
v1.eval(); v2.eval()
compare_rows = np.random.default_rng(3).choice(val_rows, size=min(300, len(val_rows)), replace=False)
raw = pd.DataFrame({name: request_score(m, compare_rows) for name, m in [("v1", v1), ("v2", v2)]}).T
display(raw.round(3))

In [ ]:
from collections import Counter

step3_rows = np.random.default_rng(4).choice(val_rows, size=min(40, len(val_rows)), replace=False)
summary, reasons_all = {}, {}
for name, m in [("v1", v1), ("v2", v2)]:
    start, usable, five, reasons = time.time(), [], [], Counter()
    for k, r in enumerate(step3_rows):
        b, y = int(lib.loc[r, "bedrooms"]), int(lib.loc[r, "balcony"])
        maps = lc.generate(m, sil_u8[r], conds[r], n=30, seed=k, device=DEVICE)
        options, why = lc.design_from_maps(maps, sil_u8[r], b, y, lib.loc[r, "m_per_px"], rules)
        usable.append(why.get("ok", 0)); five.append(len(options) == 5); reasons.update(why)
    summary[name] = {"usable pictures (of 30)": np.mean(usable), "requests with 5 options": np.mean(five),
                     "requests with no option": np.mean(np.array(usable) == 0), "seconds per request": (time.time() - start) / len(step3_rows)}
    reasons_all[name] = pd.Series(reasons) / sum(reasons.values()) * 100
display(pd.DataFrame(summary).T.round(2))
print("What happened to the pictures (% of all):")
display(pd.DataFrame(reasons_all).fillna(0).round(1).sort_values("v2", ascending=False))
V2_IS_BETTER = summary["v2"]["usable pictures (of 30)"] > summary["v1"]["usable pictures (of 30)"] and \
               summary["v2"]["requests with 5 options"] >= summary["v1"]["requests with 5 options"]
print("\nv2 is better for the app" if V2_IS_BETTER else "\nv2 is NOT better for the app: keep v1")

# 6. See it: the same 4 requests, v1 and v2

For each request: the real plan, then 5 options of v1 (top) and 5 of v2 (bottom), the best of 30 after step 3.

In [ ]:
from matplotlib.patches import Patch

LEGEND = [Patch(facecolor=lc.COLORS[t], edgecolor="#3d3d3a", label=t) for t in lc.ROOMS + ["wall", "window", "door", "entrance"]]
demo_rows = step3_rows[:4]
fig, axes = plt.subplots(8, 6, figsize=(30, 40))
for i, r in enumerate(demo_rows):
    b, y = int(lib.loc[r, "bedrooms"]), int(lib.loc[r, "balcony"])
    for j, (name, m) in enumerate([("v1", v1), ("v2", v2)]):
        row = axes[2 * i + j]
        row[0].imshow(lc.to_color(maps_u8[r], sil_u8[r])); row[0].set_xticks([]); row[0].set_yticks([])
        row[0].set_title(f"REAL · {b} bed · balcony {'yes' if y else 'no'}" if j == 0 else f"{name}", fontsize=12)
        maps = lc.generate(m, sil_u8[r], conds[r], n=30, seed=i, device=DEVICE)
        options, _ = lc.design_from_maps(maps, sil_u8[r], b, y, lib.loc[r, "m_per_px"], rules)
        for c in range(5):
            if c < len(options):
                lc.draw_layout(options[c], row[c + 1], f"{name} · option {c + 1} · score {options[c]['score']:.2f}")
            else:
                row[c + 1].set_axis_off()
fig.legend(handles=LEGEND, loc="upper center", ncol=len(LEGEND), frameon=False, bbox_to_anchor=(0.5, 1.01))
plt.tight_layout()
save_fig("71_v1_vs_v2")
plt.show()

# 7. Save v2 for the server

The files go to `OUT_DIR` (on Colab: `team4_v2/v2/` in your Drive). To use v2 in the app:
1. Download `layout_cvae_v2.pt`, `layout_cvae_config.json`, `layout_cvae_v2_training.json` and `room_rules.json` from that folder into **`01_autoencoder/models/layout_generator_v2/`** in the repository on your computer.
2. In `02_server`: `python publish.py --model layout_generator_v2` (it makes the ONNX file and checks it), then try it with `python app.py`, then push.

Only if section 5 says v2 is better.

In [ ]:
config_v2 = {**config_v1,
             "model": "LayoutCVAE2: conditional VAE; the condition reaches every decoder layer (FiLM); wider decoder at 8-32 px",
             "architecture": "v2", "weights": WEIGHTS_V2.name, "latent_dim": v2.latent_dim,
             "beta": BETA, "aux_weight": AUX_WEIGHT, "epochs_trained": len(history), "best_epoch": best_epoch,
             "check_on_validation": {"invented: bedrooms right": float(raw.loc["v2", "bedrooms_right"]),
                                     "invented: balcony as asked": float(raw.loc["v2", "balcony_right"]),
                                     "options differ (share of pixels)": float(raw.loc["v2", "different"]),
                                     "step 3: usable pictures of 30": float(summary["v2"]["usable pictures (of 30)"]),
                                     "step 3: requests with 5 options": float(summary["v2"]["requests with 5 options"])},
             "v1_on_the_same_requests": {"invented: bedrooms right": float(raw.loc["v1", "bedrooms_right"]),
                                         "step 3: usable pictures of 30": float(summary["v1"]["usable pictures (of 30)"]),
                                         "step 3: requests with 5 options": float(summary["v1"]["requests with 5 options"])},
             "v2_is_better": bool(V2_IS_BETTER)}
(OUT_DIR / "layout_cvae_config.json").write_text(json.dumps(config_v2, indent=2))
shutil.copy(MODEL_V1_DIR / "room_rules.json", OUT_DIR / "room_rules.json")

# the ONNX file too (publish.py makes it again on your computer; this one is a check that the export works)
class Decoder(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, z, s, c):
        return self.m.decode(z, s, c)


cpu_model = lc.LayoutCVAE2(); cpu_model.load_state_dict(torch.load(WEIGHTS_V2, map_location="cpu")); cpu_model.eval()
z, s, c = torch.randn(3, cpu_model.latent_dim), (torch.rand(3, 1, 128, 128) > 0.3).float(), torch.rand(3, 7)
options = dict(input_names=["z", "s", "c"], output_names=["maps"], opset_version=17,
               dynamic_axes={"z": {0: "n"}, "s": {0: "n"}, "c": {0: "n"}, "maps": {0: "n"}})
onnx_path = OUT_DIR / "layout_decoder.onnx"
try:
    torch.onnx.export(Decoder(cpu_model), (z, s, c), str(onnx_path), dynamo=False, **options)
except TypeError:
    torch.onnx.export(Decoder(cpu_model), (z, s, c), str(onnx_path), **options)
import onnxruntime as ort
diff = abs(ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"]).run(
    None, {"z": z.numpy(), "s": s.numpy(), "c": c.numpy()})[0] - cpu_model.decode(z, s, c).detach().numpy()).max()
print(f"ONNX check: largest difference {diff:.1e}")
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name:34s} {f.stat().st_size / 1e6:7.2f} MB")